In [3]:
import streamlit as st
import os
import sys
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

In [4]:
%load_ext autoreload
%autoreload 2

# Get the absolute path to the directory one level above the current notebook
root_path = os.path.abspath(os.path.join('..'))

# Add that root directory to the Python search path
if root_path not in sys.path:
    sys.path.append(root_path)
import src.models.work_experience as models

In [5]:
# test local model import
try:
    test_job = models.WorkExperience(company="Google", role="Dev", years="2020-2022")
    print("Import successful and validation working!")
except Exception as e:
    print(f"Validation Error: {e}")

Import successful and validation working!


In [6]:
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("Missing GOOGLE_API_KEY! Get one at https://aistudio.google.com/")
else:
    print("GOOGLE_API_KEY loaded")

GOOGLE_API_KEY loaded


In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key,
    temperature=0.0
)

In [12]:
# Generates JSON schema

import json
json_schema = json.dumps(UserProfile.model_json_schema(), indent=2)
# print(json_schema)

In [14]:
# Generates example JSON
import json
from src.models.work_experience import UserProfile, WorkExperience, Education, Project

# 1. Create Example Data using your classes
example_user = UserProfile(
    history=[
        WorkExperience(company="Tech Solutions Inc.", role="Senior Developer", years="2021-Present"),
        WorkExperience(company="Startup Hub", role="Junior Dev", years="2019-2021")
    ],
    studies=[
        Education(institution="University of Technology", degree="B.Sc. in Computer Science")
    ],
    projects=[
        Project(name="AI Resume Builder", description="A tool that parses messy text into structured CVs.")
    ],
    is_valid=True,
    missing_info=[]
)

# 2. Convert the example to a pretty JSON string
# we use model_dump() first to get a dict, then json.dumps for pretty formatting
example_json_str = json.dumps(example_user.model_dump(), indent=2)

print(example_json_str)

{
  "history": [
    {
      "company": "Tech Solutions Inc.",
      "role": "Senior Developer",
      "years": "2021-Present",
      "description": null,
      "skills": null
    },
    {
      "company": "Startup Hub",
      "role": "Junior Dev",
      "years": "2019-2021",
      "description": null,
      "skills": null
    }
  ],
  "studies": [
    {
      "institution": "University of Technology",
      "degree": "B.Sc. in Computer Science"
    }
  ],
  "projects": [
    {
      "name": "AI Resume Builder",
      "description": "A tool that parses messy text into structured CVs."
    }
  ],
  "is_valid": true,
  "missing_info": []
}


In [16]:
sys_msg = SystemMessage(content=f"""
You are a Career Data Architect. Your job is to map structured JSON with information about user job history to CV in Markdown file format.

### INPUT
On input you have JSON with following schema and example value.

#### SCHEMA
On input you get JSON with schema:
{json_schema}

#### EXAMPLE
Example input JSON:
{example_json_str}

### RULES
1. **Strict Markdown only**: Return ONLY raw Markdown. Do not include markdown code blocks or conversational text.
2. **Include only user real information**: Generate CV based only on data coming from input JSON. You are not allowed to add anything new or to change any information.
3. **ATS**: CV must be easily readable by ATS systems.
4. **CV format**: You are free to chose any style you want, but it must be Markdown and be easy to read by humans, clean and esthetic.

""")

In [17]:
ui_data = """
{'history': [{'company': 'NexaCore Systems',
   'role': 'Senior Software Engineer',
   'years': '2022-01-Present',
   'description': 'Architected and maintained high-throughput microservices for a fintech platform using Java 21 and Spring Boot 3. Led the migration of a legacy monolithic dashboard to React, improving load times by 40%. Mentored junior developers and implemented CI/CD best practices using Jenkins and Kubernetes.',
   'skills': ['Java', 'Spring Cloud', 'PostgreSQL', 'React', 'Docker', 'AWS']},
  {'company': 'VeloStream Solutions',
   'role': 'Software Developer',
   'years': '2019-02-2021-12',
   'description': 'Developed RESTful APIs for a logistics management system serving international clients. Integrated third-party payment gateways and real-time tracking features. Contributed to the frontend transition from JSP to Vue.js, enhancing the user experience for the dispatcher portal.',
   'skills': ['Java 11', 'Spring Security', 'Hibernate', 'Vue.js', 'Redis']},
  {'company': 'BlueBrick Technologies',
   'role': 'Junior to Mid-Level Developer',
   'years': '2017-08-2019-01',
   'description': 'Promoted from Junior to Mid-level within 14 months due to high performance in feature delivery. Built and optimized database queries for a large-scale e-commerce engine. Gained initial exposure to frontend development using Angular for internal admin tools.',
   'skills': ['Java 8', 'Spring Boot', 'MySQL', 'Angular', 'Git']},
  {'company': 'SoftStart Innovations',
   'role': 'Junior Java Developer',
   'years': '2016-07-2017-07',
   'description': 'Collaborated on the development of internal reporting tools following graduation. Focused on bug fixing, unit testing (JUnit/Mockito), and documentation. Participated in daily Scrums and learned Agile methodologies in a fast-paced environment.',
   'skills': ['Java', 'Spring MVC', 'Maven', 'JavaScript (ES6)']}],
 'studies': [{'institution': 'Silesian University of Technology',
   'degree': 'Bachelor of Engineering in Computer Science'}],
 'projects': [{'name': 'CV generator',
   'description': 'A side project written in Python that calls a real LLM to generate different CV structures and text.'}],
 'is_valid': True,
 'missing_info': []}
 """

In [18]:
user_msg = HumanMessage(content=f"Please process the following UI inputs:\n{json.dumps(ui_data)}")  

In [20]:
from IPython.display import display, Markdown

response = llm.invoke([sys_msg, user_msg])
json.dumps(ui_data, indent = 2)

'"\\n{\'history\': [{\'company\': \'NexaCore Systems\',\\n   \'role\': \'Senior Software Engineer\',\\n   \'years\': \'2022-01-Present\',\\n   \'description\': \'Architected and maintained high-throughput microservices for a fintech platform using Java 21 and Spring Boot 3. Led the migration of a legacy monolithic dashboard to React, improving load times by 40%. Mentored junior developers and implemented CI/CD best practices using Jenkins and Kubernetes.\',\\n   \'skills\': [\'Java\', \'Spring Cloud\', \'PostgreSQL\', \'React\', \'Docker\', \'AWS\']},\\n  {\'company\': \'VeloStream Solutions\',\\n   \'role\': \'Software Developer\',\\n   \'years\': \'2019-02-2021-12\',\\n   \'description\': \'Developed RESTful APIs for a logistics management system serving international clients. Integrated third-party payment gateways and real-time tracking features. Contributed to the frontend transition from JSP to Vue.js, enhancing the user experience for the dispatcher portal.\',\\n   \'skills\': [

In [22]:
from IPython.display import Markdown, display

display(Markdown(response.content))

# Work Experience

**Senior Software Engineer** | NexaCore Systems | 2022-01-Present
* Architected and maintained high-throughput microservices for a fintech platform using Java 21 and Spring Boot 3.
* Led the migration of a legacy monolithic dashboard to React, improving load times by 40%.
* Mentored junior developers and implemented CI/CD best practices using Jenkins and Kubernetes.
* Skills: Java, Spring Cloud, PostgreSQL, React, Docker, AWS

**Software Developer** | VeloStream Solutions | 2019-02-2021-12
* Developed RESTful APIs for a logistics management system serving international clients.
* Integrated third-party payment gateways and real-time tracking features.
* Contributed to the frontend transition from JSP to Vue.js, enhancing the user experience for the dispatcher portal.
* Skills: Java 11, Spring Security, Hibernate, Vue.js, Redis

**Junior to Mid-Level Developer** | BlueBrick Technologies | 2017-08-2019-01
* Promoted from Junior to Mid-level within 14 months due to high performance in feature delivery.
* Built and optimized database queries for a large-scale e-commerce engine.
* Gained initial exposure to frontend development using Angular for internal admin tools.
* Skills: Java 8, Spring Boot, MySQL, Angular, Git

**Junior Java Developer** | SoftStart Innovations | 2016-07-2017-07
* Collaborated on the development of internal reporting tools following graduation.
* Focused on bug fixing, unit testing (JUnit/Mockito), and documentation.
* Participated in daily Scrums and learned Agile methodologies in a fast-paced environment.
* Skills: Java, Spring MVC, Maven, JavaScript (ES6)

# Education

**Bachelor of Engineering in Computer Science** | Silesian University of Technology

# Projects

**CV generator**
* A side project written in Python that calls a real LLM to generate different CV structures and text.